# TinyForge-Vision: training on Colab

Trains SigLIP-B/16-224 + SmolLM2-135M on the preprocessed Cauldron, then scores it on MMStar. Run the sections in order:

1. **Setup**: GPU, Google Drive (checkpoints survive disconnects), repo, packages, logins.
2. **Checks** (minutes): tests, GPU attention check, overfit one batch, a 100-step run.
3. **Training** (hours): the full run, and how to resume it.
4. **Evaluation**: MMStar over 3 seeds, plus a few generations.

Needs an H100 (or A100) runtime and two Colab secrets (key icon on the left): `HF_TOKEN` (read access
to the private preprocessed dataset) and `WANDB_API_KEY`.

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
# Edit these two before running.
REPO_URL = "https://github.com/JuaniLlaberia/tinyforge-vision.git"
CHECKPOINT_DIR = "/content/drive/MyDrive/tinyforge-vision/checkpoints"

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!git clone {REPO_URL} /content/tinyforge-vision
%cd /content/tinyforge-vision
!pip install -q uv && uv pip install --system -q -e . pytest

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

import wandb
wandb.login()

## 2. Checks

Fast tests first: all must pass before spending GPU time (rerun them after any code change).

In [ ]:
!python -m pytest -m "not slow" -q

**GPU attention check.** On CUDA the language model uses the original attention call (padding mask +
`is_causal=True`), which the CPU tests cannot exercise. This runs the 222M checkpoint on one masked
batch in fp32 and in bf16: both must run, and their losses must agree closely.

In [ ]:
import torch
from tfv.data.collators import VQACollator
from tfv.data.processors import get_tokenizer
from tfv.models.vlm import VisionLanguageModel

model = VisionLanguageModel.from_pretrained("lusxvr/nanoVLM-222M").cuda().eval()
tokenizer = get_tokenizer(model.cfg.lm_tokenizer)
samples = [
    {"image": torch.rand(3, 224, 224), "text_data": "Question: What is this? Answer:", "answer": "A cat." + tokenizer.eos_token},
    {"image": torch.rand(3, 224, 224), "text_data": "Question: How many people are standing near the red door? Answer:",
     "answer": "There are three people." + tokenizer.eos_token},
]
batch = {k: v.cuda() for k, v in VQACollator(tokenizer, model.cfg.lm_max_length)(samples).items()}
args = (batch["input_ids"], batch["image"], batch["attention_mask"])

with torch.no_grad():
    _, loss_fp32 = model(*args, targets=batch["labels"])
    with torch.autocast("cuda", dtype=torch.bfloat16):
        _, loss_bf16 = model(*args, targets=batch["labels"])
print(f"fp32 loss {loss_fp32.item():.4f} | bf16 loss {loss_bf16.item():.4f}")
assert torch.isfinite(loss_bf16) and abs(loss_fp32.item() - loss_bf16.item()) < 0.1
del model
torch.cuda.empty_cache()

**Configuration.** Defaults are in `tfv/config.py`. Write the budget down before the
full run (samples, steps, GPU hours). Batch 256 = `micro_batch_size` x accumulation: keep 256 on an
H100 80GB; lower `micro_batch_size` (128, 64) if memory runs out.

In [ ]:
import dataclasses
from tfv.config import EvalConfig, TrainConfig, VLMConfig
from tfv.train.train import train

vlm_cfg = VLMConfig()
eval_cfg = EvalConfig()
train_cfg = TrainConfig(
    checkpoint_dir=CHECKPOINT_DIR,
    micro_batch_size=256,  # batch_size stays 256
    num_workers=os.cpu_count(),
    eval_interval=250,     # Each eval is roughly a minute
    mmstar_limit=None,
)
print(train_cfg)

**Check 1: overfit one batch.** The loss should fall close to zero within a few hundred steps; if not,
a module or the labels are wrong. The first data load also downloads the preprocessed Cauldron
(~32 GB) into this runtime's cache; later cells reuse it.

In [ ]:
overfit_cfg = dataclasses.replace(train_cfg, overfit_batch=True, max_steps=300, eval_interval=0, mmstar_limit=100,
                                  log_wandb=False, run_name="check-overfit", log_interval=25)
result = train(overfit_cfg, VLMConfig(), eval_cfg)
print(f"Final overfit loss: {result['train_loss']:.4f} (expect well below 0.5; if it stalls above ~1, check the modules and labels)")

**Check 2: 100 steps.** Training and validation loss should both fall, and the sampled answers should
start to look like answers. Note `train/peak_mem_gb` and `train/samples_per_second` in W&B to size
the full run: total samples = 5 epochs x ~1.65M.

In [ ]:
check_cfg = dataclasses.replace(train_cfg, max_steps=100, eval_interval=50, mmstar_limit=200, run_name="check-100steps")
result = train(check_cfg, VLMConfig(), eval_cfg)
result

## 3. Training

The full run. `latest.pt` is written to Drive every `checkpoint_interval` steps; `best_mmstar/` and
`final/` are saved models. Fix `run_name` so a resumed run finds its directory.

In [ ]:
RUN_NAME = "tfv-full-v1"
full_cfg = dataclasses.replace(train_cfg, run_name=RUN_NAME)
result = train(full_cfg, VLMConfig(), eval_cfg)
result

**Resume after a disconnect.** Rerun section 1 (setup) and the configuration cell, then this cell: it
continues from the last `latest.pt`, same data order, same W&B run.

In [ ]:
resume_cfg = dataclasses.replace(train_cfg, run_name=RUN_NAME, resume_from=f"{CHECKPOINT_DIR}/{RUN_NAME}/latest.pt")
result = train(resume_cfg, VLMConfig(), eval_cfg)
result

## 4. Evaluation

MMStar over 3 seeds (answers are sampled, so scores move between runs). Compare
with the published checkpoint scored the same way (`--impl v01 --model lusxvr/nanoVLM-222M`).

In [ ]:
!python scripts/eval_mmstar.py --impl ours --model {CHECKPOINT_DIR}/{RUN_NAME}/final --seeds 0 1 2 --output {CHECKPOINT_DIR}/{RUN_NAME}/mmstar_final.json
!python scripts/eval_mmstar.py --impl ours --model {CHECKPOINT_DIR}/{RUN_NAME}/best_mmstar --seeds 0 1 2 --output {CHECKPOINT_DIR}/{RUN_NAME}/mmstar_best.json

In [ ]:
# The example image is not in the repo: fetch nanoVLM's (or pass --image with your own)
!mkdir -p assets && test -f assets/image.png || curl -sL -o assets/image.png https://raw.githubusercontent.com/huggingface/nanoVLM/v0.1/assets/image.png
!python scripts/generate.py --checkpoint {CHECKPOINT_DIR}/{RUN_NAME}/final --prompt "What is this?"

In [ ]:
# Optional: publish the final model (private).
# from tfv.models.vlm import VisionLanguageModel
# VisionLanguageModel.from_pretrained(f"{CHECKPOINT_DIR}/{RUN_NAME}/final").push_to_hub("Juanillaberia/TinyForge-Vision-222M", private=True)